# Chapter 4: reproducible copy-trading risk analysis

This notebook reproduces the three quantitative findings reported in Chapter 4
(Figures 7–9) from the fixed OKX copy-trading snapshot dated **2 July 2026**.

**Publication boundary.** The input contains 200 partially masked trader aliases
(for example, `Gl***son`) and excludes platform account IDs. The public `pnlRatio`
histories are treated as a return-index proxy rather than audited equity. The
analysis is descriptive: it cannot identify liquidation, deposits, withdrawals,
follower-level losses, prediction, or causation.

Run the cells from top to bottom. The final cell saves the three figures and an
anonymised derived CSV in the notebook session.

In [ ]:
from pathlib import Path
import base64
import hashlib
import json
import math
import urllib.request
import zlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.unicode_minus": False,
    "figure.dpi": 120,
})

## 1. Load and verify the frozen anonymised dataset

The SHA-256 check prevents a changed dataset from silently producing a different
result under the same notebook link. When run from a local repository checkout,
the notebook prefers the adjacent data file; in Colab it downloads the same file
from the public GitHub repository.

In [ ]:
DATA_URL = (
    "https://raw.githubusercontent.com/risk-ledger/"
    "risk-ledger-dashboard/main/"
    "chapter4_copytrading_metrics_anonymised.json"
)
EXPECTED_SHA256 = "bf335647f2ee59040e370adb5f4b99f26d75c72a2c9ed6c36e2c63a311786dc6"
LOCAL_DATA = Path("chapter4_copytrading_metrics_anonymised.json")
EMBEDDED_DATA_B64 = "eNq9XVuTXElxfudXdOyTveycqKyqrIv9gqS9sDd2LQl2gSAUremW1Kine9zds9JA8OYI44AADBg/cPMDBA47sME4wsRiB3/Dfl9W8MZPcOYZXeYcZZ6qPrNiNlYhTfdMd3ZmZX6Z+WXWlz82mTx3NN9NZ9Pd9Lm/mHyZ/k3f2S12yzn987krd6bHu/lm4ieH6+PTg91mOlusbk/oJzaLw+3kz6ar9er0aLGdz/78uRfOfna7mh5v76x3N+hXtr/DGhsOTDww9vFTpkfHy/mN2fzWYrXYLdYrftrLi812N7HGTJbz6WzCLzXfbCfbk82t6eF8NjlZ0b8nuzvzyVuvvzuhn52eLHeT9XvzzXuL+b3JZrq6S2/t0UscbxbvTQ9P+Re/fXJzuTic3JmuZsv5djLdzCfH081uMV0uTydH0+3d+ewvJ8fL6e7WenM0mR4erk9Wu8mrL549dX7/cHkym8+aR795uTha7Kb8rrf826/TGzo+e4Xj1fIqPzDZzjcLeqUF/YbJZr472awOFvTu70+ON+v7py9MVuvdZHoyW+xIrPlfnyx2p83k6nxL8py95my+PdwsjneL9+YTeteT2br9kcVsvtotbp1Olgv6oVn7Hl6YHE6P6f0sJ7eW63vbFya31kv6C31Sy/V2O6dvHG/ms8Xh2XPXG3r6ybb9yeY5kucrLNRzm/nhejNjcT7fynhmBWwHrRJuTJeLaSvs6+vnn3/++vr44WdBz3j0cvww5tSYx4/MFlv6UE/nsxub9YIezU02Pj5++Gg2u9F+HPSQaUJImVSfozGYINrHT6OfvRFn9JwDZ3ODne87ww84G5oUOg/k9oHVyXLZfvcrLwxL9QpLdeOVV2SprBsQit85aEK57A0EZxCSI6F6b/KhVLkBK0gVGpvPf3kUJAyhgVgl4ktLEjGkpIhoB0SEBuhL05u3YAM4dD4Yb9LTErKA4bwkkg7BxQajICF9fI11VSK+viIR1yczWUQwMCijs+eM7ryM53/ssVDRN6TbJ18gyDT8nCw/p0pSc58kne/uKMocNFfrwSu6hGA9P0xHMdFXlMzVdN+vkUwXUoMdjUuKpcNcKe3xgqQ9+dJCljbjsLgOkiKuD0Bn13lI6IxDweWQsO6cGPQ8QVofms5BFV2Rc43rfCRVkn9ySpJT9JUl93FQchLcaYq2FjCGFPjIorMoKtq5zqntyfVQ1diErp+ShIcmuY7RVEn/9oakj8HI0ruCmTtnFelNsNFl9M6b7B1E4XQ3YHShziS30ES8SNB5k5UbQlDcVR6OOin4rNl1zMEFS2HUAP1NcMkHQNo1+gl1j05ohvNq84K4maKDr5LXRZKXvqW55zQor0mou61krEk+O2PJd1kvCRybkAQZWVsXUuIJC/UYdfZPqC/5Js1GAUIkkcCjySH7/LRIZP75vHIk1wT0pI63lkKQJQeKdTCJMcRqMVXAnxkUFlLSFGgNkLNCkxx4Hz2I3ihlESSB5G0pAKU6o3xpzpF0fl/xMcMiebTaGbQZCAyz5rzNxgn6O7CxF13EM8h+/LwGJXE90ut1nlQl+1+R6H/88dd/pKAIzIN4CcFq+nQeM8OpHDM7ISm6uNBIBpt9k6yKLh7aq/NNOC9uHTq8S9JuFVFhOJCGqAXS5HICn1MmKBGDd5KaE6U9hVjibYPp/HMkLYfc+P2h4qUtyb1YNQp4GnZQgAEHUhvjEp0AT29ddLqxqyaQJAcgGFaKMRSIwO0t+YucqY5CyOSqNLeMiKRmAylmF4JDSWrsCGRQBg+9VCk/+n5KVdLd4cATtXxu2KANooqNMEVLR5/VGjgdkM5ugx1vI4eexl8krl7bkXib7U4JNWkwl0tNcF5zTpGgUYyEfzEm452U42Skd2+Gjyy4FBo8dx4l4IsU0ZpY542vr1skoUgMgMP4yNFZ1GIRxdToCQ8HT0mdCcJRpY/zvNFCQDHYxo4j83Kek+rAxGdfInlf/dQlxTENH9HoUM3qgL0kwcDkHZ1WAd2jJTCSS+ptfCmtiXQQ6g7rbYb690+fV7xRHIL69EHEc1G/WJk4SJZQERQy9JDokMjQKVNa6fbO1Rga0jc1LDGsTnq7KjZ0FFsMndRACVt00nk1HXHJeqOUqcYushChBeFlNyJRvblrY6ySBAxn6TaEoFkyBkJPgZIEm+15gz+X1ngCWzpwcI/Sbx8Lkkc+OHuH2De5MgNOc1nDUYh8r+axvA+R0h8wkZwW5XRPS06+F87LbSWHdWDphLpSBc71YHiV5G+w5Iv7Sk0qDOvcn7fQfipEMQUTOWpOAEGCzoAUf0vFCcrpXAlQ+urYBKxmrU4xLCtaFWskpCgNyRLQ8MZHKUfiBDZ2imtylpt86WiTB8K6ROF3P/2fD3/wfZKYXk450WYwM+IuhgY+vEtkbTaRLwoo2DXppJzVJwpOXqyVx95P16X1XGe7dUspzAynwEiZnqZeMmJ0DgKlwr6PFh4eYhdNoYhOH4grWjIineH6es3LayXfp8g5XMIwWhsEjYNsyVvHbHySaopcZSlZMtu7LZbcCN+aDhCrc1mffIdPsTFj1AwUgzRfTTpMFKGysZQEG7l4Y7vlVCMJT1l90E0/P0oXu59Pndbvs7f+0nStuLACss5BM3GHPkBOYCHSH1GyccqWuh7MSyc69qKvpPaUm2z3biF8nptiy/kXxmjdZdQiNDh7Zu/IJUpnJXu3/hkUXD/H9YyPf3yUOEHN6zE4m2wM2aHxXuw+237nS0yRuKt0oU401yMXCwVXYOGQ6i0PbwJw/mctOI5BQcoBkymm9YSiYaC68fiU+joA+amzpHc6pooBEbTqK2STQ/Q2IpeoknIsBfGwCfEi6nuR1XdzvR7nY40mj4kQMYLNESi40P+yk+3ESdnT+G4lTUTE1dJe5WC63s5H1aCCC2oNKiRvPVLQIYRI8VSq0VDE77jVLFcsQGRNNPv0X09249QZIajAl45hQkRIBJGsE5ERyuaZdJnzo1gCHaCxBz9ktduOKZiaoIJA4yKhe0rgyeyizVKF3DaxI5SYwxJ6l1B9pvy57mAyxl1pjqbQSwYEVANHdoF8K8UPks8KgcNCE0Soh40TCzBIr7d/DaKFOGrGYguRg6CpBu+sayNGpBCSzwfQJ/yeJkglJj4BnUghsnvoMFTWWG6SfEnlLxU6dc6rSCa4yCkZwTdDJurGZSnWNVFypc5ThtTB81XCHt5ha71tFcc62Jnj/kVQwXoKicJmoFjD4E1MufvlfSv3L4KV+xehTp9XOPLfXq/GuFYTNBxnbcIYYiCvyrS7LJor2pI6Kcvo2q4Vo0gfCtWVF37xTZL8i4odFxgDdFDVQimlXxRUMVvPbLwoxkxni9wt7kwV66Tk+Lrl8boWx6alM221StKw6FZNwyi7pOyb40wmLyzCvdCjsiDKZCYTSvGV8oBUF1JfY3i0nN7T6BIFIz9PpuwFVUq3QzRIinKeG1WSxLYBU1R2D0QlKe8ki6wrMCw5BC1PRqEli3oEopOcMNPnlQkNeiNYtguNEcug/fJ4kvkvOKJqdpmh/ptTpZIEYdiajVMhEyTy0oSbKOb6zPREsV3XEQu9CJm66UB0onJ9p6ZQ58Pe4fi0ONLA4rAPoyQ6qfWUKLV3jNjBItVKAplac32JSbRH86UCCV2hhu8i1DflyDtRZnku6Ei5NSWr3bgjyYf0m+py67fZQJfvvjsqeUlWw0sxi4QesE0o1je5dYHF4JIa6/cHwFeucClhsdq/ckKpc6fk1qd8JJsgkj34wLlzkizUYbFyQq/Sy3ME2dtSaecL6oz5Xfa9b8yVVNwXEhwDamGbgCIlNh49OWHDjTIJIGeZANLVoljKfupJdVX8+RDr0hZ0nVBLzB3BCCaaUrYTIzoUDJ25Z8Uyvu2rUXK9B2kUiPrfn/3f3/C0Svb7l10OBmEFBBY8pZwc/Sdauuu1HvMzqPBuPYmHSptiuFTPvVPNbRE0pizdkZMi9IQoN9MRSpDJpSaHwikO2NSd2tfYjk8//OEvHvzwt6NiKcEAlUHsDCUE1gVrjY8ixR/6jgtkv9WlV8jW7Jtc56cf/POvSOg/fPuX+7MlWvNFUCm3yVtgdJESok0o9SR6kEiS13XZEjLl9oBtpc6cT4bIIYUcIKphOHhuOPqUbLCU2iQJJKIteSpsfB5Iix4yJAidm725IVfbToaW4A+y2djkjE6LCeyfPDP+6W82CpAxi5CxZU6Inph5P3F/V3yNYfBmrZH/TaGbHK2eAqCJgIQ42uk6cRArNq6Tp4u8YtdAlll6tfTh2aCIYdh+s65FQsQpxxCdpSDoQeqY20jooFQYTtjT9Z7R5vJ0iBDgC2ZqrTqFgyFkCjYeckQLUcrimD8uwgdCwFKmemAD5R11jbXpBbwOoDrQ2m9NPwwkPUKCDP6b0AFFPomAEDvO19eVWi4dMSC8rSWkYfgkBpeyKjA5xUCnMTGPxQtWClxEKaEG9E260LDY5zi7IUSqmGnBmVq9+k0nDHlA+SyeoITuuyFCjJmpCaZE0mEv2wW8vr5qONeGIAv0pKCzopEiJ2CG7Aw3GSW79r5/EJ/wdMThP67Mdwc5qkT88B//6cN/+DrDov/4yR9++d/jgL0xakOVYSBax1X/SPg+GCc2kE0X5wEoeazp4iOjZLKmlytUjmRdf2sIOBRcl1VnXk0g32UiWSAmD5zOixXyFJ51fnOTGSz3tbnPgnwuoloUNkB5ejKRoAyJKkVUdtRinbRJIGJ6i/WjrDnZUc1jr+4WMJgSxOhdSIZ8n2K1veREERCLU8q+P/RbP6hyRbPW0hyOVcsuiVRIHjkZwlDZgsT4yOIIne2OcsgMHezyzSrBYIt3F7tmXNlFnxdEMgEIyROqxxi8nKnKYxqmBypADDyN7Wg/1mXnL3NNbXp4d1yLUnXGFHMQU0TyxonwTYpSzO2qUOSaXdQPteyd6UZbl5EL7KucQK2iJfDBPfrfSkU0V3SzEBpwF5EPrjFu2MD+6jsYrIgm3h8QfSJU5B05PTE3gyKjhT1XFw1hko23j62rpP/0tm0+Ho8yXqNzs7wNHgNlpyZSQAWpRAjPYtAeEsnjAEdqU2+lRsftcl5hkxJm5+T5onLdyPQyF80VpTHavPw6SW9GmrJxXg2wTN6NKdBxJJV6Cf/30zajGLPptkCMOKFwwMMvHfGhDmFcZ1/82cVqHDQ+j2T7+qdkFzKyUQcTfERJ/xFlmYOi48qdTJevswfWOpCh1LOKKv0VrHEkDxBuIJ1K/TrbK61kJ5dCQ9emRZ3Gpg4qvboaKrUUXFJyOlRKyXqwFEzBe4nTJHeQD0wfFEqoMDWuctycLfTufDX2jOoMSkpWyNtShm5dClGbgKyobJseDpL5EO3mn+5ZrjPnBdcKV7NxR9QZld1DmNAmXq8QOSEQHDRYUcGhAvSTiXfZspV02XY30XaxHJfxqElquy5EBQ4P/bHvVuPFkwu9qo2VmT29kdg6AttPfvbgG19j7sR6O0rXpEw1E4gpR8yJWQIhZtHUQ5eeJpca7YWmMO6eVU+VYluBpxaz5qkipXJM2jIZeDAuyVMmHmTmRIYicyL2FzbVKfTnP/n9V39MEn/w6+89eP9ftP1MhSIjJapeb2d4aziPd5lyWYlkSyij04CxGmcaykeaqShQv9CIF52O89lB397EbBEe83WGThiK65uyK84vSHQQ2WPHrseG+gR3rpHih50YhSLNiUXrmC5CfxiKSuL0hg+9Vrs47RAJeBWUzQuDzN5Fmzb1XYxUO6LatssObOQpSHpPKQvJUerDfwVXmmIF5wBr55mvHXLuYEdmTtGrHTxmY1qKIIlswUZ5f+JTSEISmDxL+VTHxtcV069waL6j9QsKodmqMARMRmiHPXNkGr1IjikOCpjuYpmnFhfkRzU8t78f/+D935Lkv//Pfx0luVOJFZADs+kBKUmmbDlIQy499y3ZNSUTHZAqrnt1vb0OWN8iWoyp1LXZ4QCnno50Rus9BWyCbYLkkJr8LKpzb5+0jfd740RK+n4J8M76HAzz25y0tsvxKSj5ZtPrc8mZROpuANtnq8jk2jjJbYoqPyhzDRoS0Fu3MghxIKfB4hjhgW3SCNbIJU6Dp1/cjJPPqd0SC5YMFcnT8+pIqewa5UktHo7wFYnSHptRpmOYB2edarWqbKPJXKuCZDNvbhUn0XCgXPFYWos1GBJH7Fg4YuGbzcmoCRfrrD4ATHkDBALhTEKOkPegyXczSnGhIqVk+w/zsKhrTc8uFBStw0dK86OlJIpOco4gTfOAG9zl9ChZCI0vDnEdkE6wDld8kifWbIyj+PQI+rJMPrOJ+Rc8KitvZ7Ox279S7DqWWJq8TrQuy/8Sx5/g7KhjbL1T17PFYLyN5IjJXyUH4uKI8qKU0NtfLB5iHv9J+0/2bI/b0LsZMUrLHjqodVl6L8CFzByz40E9CUf15iNQEV4SN/QGDOrM+hLnwBttqW8u0TTjQNIfIwVcclnOp+iivF69NDbhe9PQYgUaKHLn/SNxu0ZfnSnGEhVD316GmdfecpMwiq373N9YJ1Jzxfmm/l7CWGfTM65oHa6PFG9dIl2rLHMfea8kn1fvs5EuuMi9SqSXEZa1NZzrDPsb+NmK7qP5yMEgNSQTiuYNbZSvUIANKNfluzNRytKnPvHKyjWeXEnZ3Q3lv5TslOp4WR3PC3xjAmVxZA/eiHwc19sRKpe1cpMvkhsdtms5TBoVmyjuqMzyTHDD8fZ8b00IVtw7CL44McGXspTYkPW1aH33ni2IyrvXtEK7jQSvgDKiiNl4cTqkt5lZQVl0KIvz4tXCXhpMHrDAWoia4QYOQo4Uj5YXS3qpMCfe7eF6mhRvMjFN7MKz2nLN+w++y2X333376w9+850Pf/SDEXGonc5RL1XirihamxLZcrLSTRC8UC0qDNcUy2kSG8n+gbede1qvxhVgIaucXt77QDEXHKGoYCQ9M+feK3bcXQqclGyhcr1xywu8fXs6jn+jtlIMGTBf28LDBQEhQG0iaHoLQTVTfupZdUNdf/f9lsugdX9tSV4168/MWnaUJOXEjO24h7zd/D2JFiw9re6yKTbhY433WQJUwakTFswItM5TkoTYucLnfLHVl2Os6w+XPM4S3IgT+8Z8cDt5YZ5EXZ8EmKIO+4bVK06PQHeHXV2u217/dnMzanGF0Q7qxSSh79ZX2w7GMsOsutUTeGaUmY4uEVJAFKfeAXMR3RLaBLl2Wnur29kWFe2gFeZjjXrpTATyJ5ASZeL0OUiLJ3xjFW4FxmILx/R2EleCgpYSttTAeyxwrF1SIRBSPtqumzOWB6DFjnRKpbL/gQe5Fl5/YRInJ4ouIReL+yqP0wVk3519cCFmK9I4u7UVmWHQSzHFFUDkW7tXukCdKb/WjknguLyFMI3KmiETRB8iJZsuBi9f25e6vEtRuxDlosOBwwZ9Pe7ZrUd2N0yMKm2egIljLisDAXqeGCq80lDvhbyscFO7OL52SeS6JQspzUfwUMI+sg/OPKvezrO0qGR+YERqqtJlvthiz7d4nPJtjd5VGsHKuT4mPtUm/Wi6p6+8MlTZS4XmBHM41G2BvNXcRDpAOcqbZUNS+MI5Nyi31mJTufjkGtvaej2OZBrU9hKPMmSm9LdMQS+vai9CTuubaEsZxkGwlXThTzG02R788cff+reRzkSv3Z1tUEseDYQcozJZNsyW/Wgs9cGvvsclgJ9/a9yktk5sD9lD8KzT0Ioprojr9ThRBHCpuDH4aU5qlezTWyT6PY1naQurQEJSW+GZMCuzHHi2N4m7BQB613R5bY9P8WYJPvBQV5q9vm13VM1GonUVABjvfOCVgNz8TyGItGELMuBJpkzksLXh/4ghjvM44n4FLu0EtWBJodlbvtCUTBasRDazfRaklwcGbbEKwq3j7qLlPQYGX5uejgubTp9RSsFFsudgeVjWiTggaDRwceUhefO6S7raC4hPNJ53yWJV7iCPz6FPJlL+FZ20zxwomIRy9ZlXvRcPaP8ynDrZLx+wY/7FNx9842tj3fNAHc90h+idy3sUDKCqdFe5X+FNxhSvrGcjJbRDMTa5DMB7FiBYuHBFpDfjUlmbbJfFzU/HDS3rewENr8Xj9VmBJzetE69KLO1KeELz7s0OOjlBqbxq+SarVLswHaAYZzSRs+d5yUAhxmWDEnlQHbXqpp9alOnWNSsbvNfvXGDXjQn69gFSQ/I+O+bTiYzf/ey3cijhjXYj0ea9cWPn6qQrJoeJ52uMI6crFUgg9+fkpX1LDbhS/Hyq/1kl9ynLfft0Ma46ktRrxjBaCJFwPgmNQVr7YXsUBafE0q7k4vWJBy4RXq6/K0Q128LN4RBV6Bswkl+i4JmSN9VH9OnZIXm9e+w8KdXTs1c5jYS9QxeFMNMGHcWYRNmLuAQvyu3Nfkkoyg7JYT2J6vKY0daWra0vcaQ8hrnnlFcHH1MQ1Zm7GzC82MfOuXgZQ0ujsvvD3kuDxZZilp5V9bLAPvuELhhCv/KtN73RES/P5ndHD2Rt0yc5ZhR01t5kMP/E2CEyNf5QNgPGeXJdbAZWHuKWb4kpjjW3w+tm/6N8s115eHM99iir0IIvLTB8kk2MLhtZ1zGZ8oBND3AF5WB3ucGuvguqTv2Wi9sqliT1MgOYt8yh91HsXlifK0bHtMp2HXBsr4LU872xGawFH9rmC51P65x31VGpP+8Z5Ww2jTi3n5kO3T1WVKZ+hXpiOiDDYwu8ueqCeU/9Jsu2UboYKQ+qjVJw3eTES53SJKbn2Ig3SrTfr+uY8XkbZYzG6tsxIhBCsjZxbMnGXhTX29Tdmlcl2S2OGs10MXJkTV07yuOJ1oZ2v1RwVhTOdocynVe63PJyE5/q9/+q2z7I35UM0qoQ13trTTtM7k0W9/iZ3l2i8ghbbzbVRtlWu+Zfyf96fdV2akaWelHfmxsAM6855PuxxR0C+xgvQ75QdxTf4eLmyVbp3WMoza2pjCcCXZHZbLyLM4lTEaEHS+Urm3InCMiD8q53l1dw9bJfvTpOl+fHRvvDmCbTWbTBEtZDEHe0234F9wlpPMsc+WT3r3X+4W//nkeHv/+1kXcKqMHQukyu1tGJdYFScNnZonw8Ecsw1jdo693R9s5i1AJdOm8qQTEw5Qx5ARFCdF4c33K5fD0I9Pm2Skaa6uLLPZaXztU4im0EdeDDRL6UKJHzocxbuQ2lYuifI1X3Q5F3aaXKbvHN9iqF9clISrG+fai9fIrbFSm4lHIUo01vGA2UJcn2YpfLDl6Wa0uTterYg80Gsos2xZgpqojDAK43SymbMPTK3VYZre0WI+oO8Ae//s0H73+DKw0YRwZVdQEp320TvWU8yElYkFJtG8orHX0TZAgR6pdKHWvraHwxI9FJ4x4wMMGDMaGy4w4VPpwFpXyA9YyVxWYkDnLqglxnIhLQpQAKgOhEt4upVC0o7Dq40Jz/m2d1oZFMOKsiQC5iY0qRh8YCL1WSrbW8dqVPi0sKUTd3flflbsO25jt5Xdni4Atjd4R/1e1CPN1hrefFM2gl5k5k6nupUgKRPiSZ6shGUVf4HBwULrojNeY4l0jH8PhPVPZed5QnL9EdvpjhsX+COpN+55SVeulVZWFlkY+lL04CTD4YSm7AY5YZDnJWg8pCrO4CHbKXOrN9hw/t5vLNUWMdfNOSuuLaWEIQhDPIkyBUzzkcuKdY6Ptep9UuklFwkbXjhhsoX+E7agCiYe/gAP804w4vHQ3xH0vSGBxaA+tT5GujyNXbCPUFSi+K098sWt9Fmv7uOz8duapLLeE5vukimRwxIC9vltmOWJ72DL3l8to+2BHL6D789++S9A+++l8jkYK6U5Is1HJ9nRJRytCChOAp96pYn8pXfinTAN2+RF1B5dK9oe17pYzFeC1Dc9F7TzkLxJQsiqMQ5XXNT6ZAZSqK9dXr2FK7SGTERpy2ra8SXD1n3mAdJZu2cxlryaOmxskL5nwtvebTHPdv39GG4VIssRzVMRaMLpOxEnrhUVYQS+1dOOZQy8TkTRmxN7Bbh+Lfeo87+NvTcTsUzl9g1lMiksslg3XRBB5GFzeiaJ4oF7etx1E3yb68GVoYUWade3UvSE5tPYxXKERI8iyr7TJRUNu23gV/XruCp3/K6+udy8XxyGFeXdvRh2QS8NBd8BcnBHaTHFdXzF1zlJ3f3e5/wSg7JIvqZiM6WC4FNIG5bilL+9dcFPv1clxBen79zNXJ7e24XU2g6gsNUpaFgdflxSTNDEJqQlSusBu6u+9Jda+SiHzp3SGOCZTrCOqynsAddz6SNkNKMmGqu4JIWblte8sOnTI62N3lVUnSPWTexcl03OAPBNVoufvQ9h4iz52LlM7Y526KAactIxZvv8j9cfsq4T9xmyGEuqeptGlO5cpFig4+eU/vvCXNiZ0nKE6JgO/dGuYV2cewIl88aVdlnH50s+lPTSw9y0G8awyVVhopASiGD9/f51TOTM7A2/IojvCVZ04MJqm4Xgz6CEGqCaGr5c9cn7bj2yN5gFEt0UeXvKNEpqUtyIvjerRcrLnxIoKyWRr3T17fmQ9Rh0YuUWDKf0xcv47OQ+f2u2daZ/gM30G+XGpmG8euM0HwTMN2mPmC+Y96JIL7N3U5GUeT2zeujGPMq7fDijsvEJ6BW2kFuHVr5AIhyHpASHTCwMfA6+mDuBy5vM0OG5+KOwT4zukx9w1y1WC2WI+7eUNl6LnMzfkAPOIBYjadKeOuWMvpy9fxWd77WCXsnfYejs1H61LaM8OZA0Xj5GJyfxqXwlS2qx+xJHTes0WXHfl7jPgnco7tVWwxxHHeQ105s8fGHDFK87fP3j/9+YWPfeVj/w+yRCMd"

if LOCAL_DATA.exists():
    raw = LOCAL_DATA.read_bytes()
    source_used = str(LOCAL_DATA.resolve())
else:
    # The embedded copy keeps the Drive/Colab version runnable without
    # exposing a personal repository address or relying on network access.
    raw = zlib.decompress(base64.b64decode(EMBEDDED_DATA_B64))
    source_used = "embedded frozen anonymised dataset"

observed_sha256 = hashlib.sha256(raw).hexdigest()
assert observed_sha256 == EXPECTED_SHA256, (
    "Dataset checksum mismatch. Stop and verify the archived version before analysis."
)

payload = json.loads(raw)
df = pd.DataFrame(payload["records"])

assert len(df) == 200
assert df["trader_alias"].nunique() == 200
assert df["trader_alias"].str.contains(r"\*\*\*", regex=True).all()

print("Source:", source_used)
print("SHA-256:", observed_sha256)
print("Snapshot:", payload["metadata"]["snapshot_date"])
print("Captured ranking positions:", len(df))
df.head()

## 2. Reproduce the reported descriptive statistics

The maximum-decline variable is already derived from the full-precision archived
`pnlRatio` histories using the bounded proxy wealth index
$W(t)=\max[0,1+\mathrm{pnlRatio}(t)]$. The notebook checks every number stated in
the three Chapter 4 findings before drawing the figures.

In [ ]:
def sign_counts(series):
    values = series.dropna()
    return {
        "positive": int((values > 0).sum()),
        "negative": int((values < 0).sum()),
        "flat": int((values == 0).sum()),
        "n": int(len(values)),
    }


def percentile(values, proportion):
    values = sorted(float(v) for v in values)
    position = (len(values) - 1) * proportion
    lower, upper = math.floor(position), math.ceil(position)
    if lower == upper:
        return values[lower]
    return values[lower] * (upper - position) + values[upper] * (position - lower)


mdd = df["mdd_proxy"].dropna()
anchor_positive = df[df["roi_90d"] > 0]
paired = df.dropna(subset=["roi_90d", "roi_7d"]).copy()
rho = paired["roi_90d"].rank(method="average").corr(
    paired["roi_7d"].rank(method="average")
)

lower_cut = percentile(paired["roi_90d"], 1 / 3)
upper_cut = percentile(paired["roi_90d"], 2 / 3)
bottom_third = paired[paired["roi_90d"] <= lower_cut]
top_third = paired[paired["roi_90d"] >= upper_cut]

results = {
    "usable_public_curves": int(mdd.count()),
    "median_bounded_decline": float(mdd.median()),
    "decline_at_least_50pct": int((mdd >= 0.50).sum()),
    "decline_at_least_80pct": int((mdd >= 0.80).sum()),
    "roi_7d": sign_counts(df["roi_7d"]),
    "roi_30d": sign_counts(df["roi_30d"]),
    "roi_90d": sign_counts(df["roi_90d"]),
    "roi_7d_among_positive_90d": sign_counts(anchor_positive["roi_7d"]),
    "spearman_90d_vs_7d": float(rho),
    "paired_n": int(len(paired)),
    "top_third_negative_7d": [int((top_third["roi_7d"] < 0).sum()), int(len(top_third))],
    "bottom_third_negative_7d": [int((bottom_third["roi_7d"] < 0).sum()), int(len(bottom_third))],
}

assert results["usable_public_curves"] == 199
assert round(results["median_bounded_decline"] * 100, 1) == 29.8
assert results["decline_at_least_50pct"] == 51
assert results["decline_at_least_80pct"] == 15
assert results["roi_7d"] == {"positive": 115, "negative": 56, "flat": 27, "n": 198}
assert results["roi_30d"] == {"positive": 98, "negative": 78, "flat": 13, "n": 189}
assert results["roi_90d"] == {"positive": 83, "negative": 83, "flat": 5, "n": 171}
assert results["roi_7d_among_positive_90d"] == {"positive": 35, "negative": 32, "flat": 16, "n": 83}
assert round(results["spearman_90d_vs_7d"], 3) == -0.324
assert results["top_third_negative_7d"] == [27, 57]
assert results["bottom_third_negative_7d"] == [8, 58]

print(json.dumps(results, indent=2))

## 3. Figure 7 — displayed return and bounded path risk

In [ ]:
COLORS = {
    "ink": "#1F2933",
    "muted": "#687783",
    "grid": "#E2E8EC",
    "teal": "#26796A",
    "red": "#B94B45",
    "grey": "#AAB4BB",
    "light_red": "#F8E9E7",
}


def style_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(color=COLORS["grid"], linewidth=0.8)
    ax.set_axisbelow(True)


usable = df.dropna(subset=["displayed_roi", "mdd_proxy"])
y = usable["mdd_proxy"] * 100
sizes = 20 + np.sqrt(usable["followers"].fillna(0)) * 8
colours = np.where(y >= 80, COLORS["red"], COLORS["teal"])

fig7, ax = plt.subplots(figsize=(10.8, 6.4))
ax.set_xscale("symlog", linthresh=0.1)
ax.axhspan(80, 100, color=COLORS["light_red"], alpha=0.65)
ax.scatter(usable["displayed_roi"], y, s=sizes, c=colours, alpha=0.72,
           edgecolor="white", linewidth=0.5)
ax.axvline(0, color=COLORS["muted"], linestyle=":", linewidth=1)
ax.axhline(80, color=COLORS["red"], linestyle="--", linewidth=1)
ax.set_ylim(-2, 103)
ax.set_xlabel("Platform-displayed cumulative ROI ratio (symlog scale)")
ax.set_ylabel("Bounded normalized decline in public pnlRatio index (%)")
ax.set_title("Displayed return and path risk are distinct", fontsize=15,
             weight="bold", pad=18)
ax.text(0.5, 1.01,
        "Platform-visibility sample; 199 usable public curves from the first 200 default-surfaced traders",
        transform=ax.transAxes, ha="center", color=COLORS["muted"], fontsize=9.5)
ax.text(0, -0.19,
        "Median decline: 29.8%. At least 50%: 51 traders; at least 80%: 15.\n"
        "The proxy is bounded at 100%; it is not audited equity and does not identify liquidation or follower loss.",
        transform=ax.transAxes, fontsize=8.4, color=COLORS["ink"], va="top")
style_axes(ax)
fig7.subplots_adjust(bottom=0.22)
plt.show()

## 4. Figure 8 — ROI signs across overlapping display windows

In [ ]:
bars = [
    ("7-day\nall visible", sign_counts(df["roi_7d"])),
    ("30-day\nall visible", sign_counts(df["roi_30d"])),
    ("90-day\nall visible", sign_counts(df["roi_90d"])),
    ("30-day\n90-day-positive", sign_counts(anchor_positive["roi_30d"])),
    ("7-day\n90-day-positive", sign_counts(anchor_positive["roi_7d"])),
]
x_positions = [0, 1, 2, 3.5, 4.5]
bottom = np.zeros(len(bars))

fig8, ax = plt.subplots(figsize=(10.8, 6.2))
for key, color in (("positive", COLORS["teal"]),
                   ("negative", COLORS["red"]),
                   ("flat", COLORS["grey"])):
    heights = np.array([count[key] for _, count in bars])
    ax.bar(x_positions, heights, width=0.72, bottom=bottom, color=color,
           edgecolor="white", label=key.title())
    for index, height in enumerate(heights):
        if height >= 8:
            ax.text(x_positions[index], bottom[index] + height / 2, str(height),
                    ha="center", va="center", color="white", weight="bold", fontsize=9)
    bottom += heights

for xpos, (_, count) in zip(x_positions, bars):
    ax.text(xpos, count["n"] + 4, f"n={count['n']}", ha="center",
            color=COLORS["muted"], fontsize=9)
ax.axvline(2.75, color=COLORS["grid"], linewidth=1.5)
ax.set_xticks(x_positions, [label for label, _ in bars])
ax.set_ylabel("Number of traders")
ax.set_title("The sign of observed ROI varies across overlapping display windows",
             fontsize=14.5, weight="bold", pad=18)
ax.text(0.5, 1.01,
        "Descriptive counts only; the chart is not a test against a 50% market baseline",
        transform=ax.transAxes, ha="center", color=COLORS["muted"], fontsize=9.5)
ax.legend(frameon=False, ncol=3, loc="upper right")
style_axes(ax)
plt.show()

## 5. Figure 9 — 90-day versus rolling 7-day ROI

In [ ]:
colours = np.where(paired["roi_7d"] < 0, COLORS["red"], COLORS["grey"])
sizes = 22 + np.sqrt(paired["followers"].fillna(0)) * 9

fig9, ax = plt.subplots(figsize=(10.8, 6.4))
ax.set_xscale("symlog", linthresh=10)
ax.set_yscale("symlog", linthresh=10)
ax.scatter(paired["roi_90d"], paired["roi_7d"], s=sizes, c=colours,
           alpha=0.72, edgecolor="white", linewidth=0.5)
ax.axhline(0, color=COLORS["ink"], linestyle="--", linewidth=1)
ax.axvline(0, color=COLORS["muted"], linestyle=":", linewidth=1)
ax.set_xlabel("Observed 90-day ROI (%)")
ax.set_ylabel("Observed rolling 7-day ROI (%)")
ax.set_title("An inverse cross-sectional association between overlapping ROI windows",
             fontsize=14.2, weight="bold", pad=18)
ax.text(0.5, 1.01,
        f"Spearman rho = {rho:.3f}, n = {len(paired)}; this is not an out-of-sample prediction test",
        transform=ax.transAxes, ha="center", color=COLORS["muted"], fontsize=9.5)
ax.text(0, -0.19,
        "Top third by 90-day ROI: 47.4% had a negative 7-day ROI (27/57). "
        "Bottom third: 13.8% (8/58).\n"
        "The windows overlap and the data concern lead-trader ROI, not realised follower outcomes.",
        transform=ax.transAxes, fontsize=8.5, color=COLORS["ink"], va="top")
style_axes(ax)
fig9.subplots_adjust(bottom=0.22)
plt.show()

## 6. Save reproducibility outputs

These files are generated inside the Colab runtime. They contain only masked
aliases and aggregate/derived measures. Colab runtimes are temporary, so download
anything you want to retain before closing the session.

In [ ]:
output_dir = Path("chapter4_outputs")
output_dir.mkdir(exist_ok=True)

fig7.savefig(output_dir / "figure_7_displayed_roi_vs_bounded_drawdown.png",
             dpi=240, bbox_inches="tight", facecolor="white")
fig8.savefig(output_dir / "figure_8_roi_signs_by_window.png",
             dpi=240, bbox_inches="tight", facecolor="white")
fig9.savefig(output_dir / "figure_9_90d_vs_7d_association.png",
             dpi=240, bbox_inches="tight", facecolor="white")
df.to_csv(output_dir / "chapter4_copytrading_metrics_anonymised.csv", index=False)

for path in sorted(output_dir.iterdir()):
    print(path, f"({path.stat().st_size:,} bytes)")